# nanoScaling Model Inference Demo

This notebook demonstrates how to load a trained checkpoint and generate text using the `ModelInference` class with KV-cache support.

## Setup

Make sure you have a trained checkpoint available. You can train one with:
```bash
python train.py configs/train_full.yaml
```

Or use a pretrained GPT-2 model directly (no checkpoint needed).

In [ ]:
import sys
import os

# Ensure the repo root is on the path
repo_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if repo_root not in sys.path:
    sys.path.insert(0, repo_root)
os.chdir(repo_root)

from inference import ModelInference

## Option A: Load a pretrained GPT-2 model

This downloads the GPT-2 weights from HuggingFace (first run only) and wraps them in our model.

In [ ]:
engine = ModelInference.from_pretrained('gpt2')
print(f"Model loaded on device: {engine.device}, dtype: {engine.dtype}")

## Option B: Load from a nanoScaling checkpoint

Uncomment and set the path to your checkpoint file.

In [ ]:
# checkpoint_path = 'out/ckpt.pt'  # <-- set your checkpoint path here
# engine = ModelInference.from_checkpoint(checkpoint_path)
# print(f"Model loaded on device: {engine.device}, dtype: {engine.dtype}")

## Generate text

In [ ]:
prompt = "The future of artificial intelligence is"

text = engine.generate(
    prompt,
    max_new_tokens=150,
    temperature=0.8,
    top_k=50,
    use_kv_cache=True,
)

print(text)

## Sampling parameters

Try different sampling strategies to see how they affect output quality.

In [ ]:
prompt = "Once upon a time in a land far away,"

# Greedy (deterministic)
print("=== Temperature 0.1 (near-greedy) ===")
print(engine.generate(prompt, max_new_tokens=80, temperature=0.1, top_k=None))

print("\n=== Temperature 0.8, top_k=50 ===")
print(engine.generate(prompt, max_new_tokens=80, temperature=0.8, top_k=50))

print("\n=== Temperature 1.0, top_p=0.95 (nucleus) ===")
print(engine.generate(prompt, max_new_tokens=80, temperature=1.0, top_p=0.95))

print("\n=== Temperature 1.2 (more random) ===")
print(engine.generate(prompt, max_new_tokens=80, temperature=1.2, top_k=200))

## Benchmark: KV cache vs. no KV cache

Compare generation speed with and without KV caching.

In [ ]:
print("Benchmarking with KV cache...")
result_cached = engine.benchmark("The", max_new_tokens=100, use_kv_cache=True)
print(f"  KV cache ON:  {result_cached['tokens_per_sec']:.1f} tokens/sec  ({result_cached['total_time_ms']:.0f} ms)")

print("\nBenchmarking without KV cache...")
result_no_cache = engine.benchmark("The", max_new_tokens=100, use_kv_cache=False)
print(f"  KV cache OFF: {result_no_cache['tokens_per_sec']:.1f} tokens/sec  ({result_no_cache['total_time_ms']:.0f} ms)")

speedup = result_cached['tokens_per_sec'] / max(result_no_cache['tokens_per_sec'], 1e-9)
print(f"\nSpeedup: {speedup:.2f}x")

## Reproducible generation with seeds

In [ ]:
prompt = "In the beginning"

text1 = engine.generate(prompt, max_new_tokens=50, seed=42, temperature=0.8, top_k=50)
text2 = engine.generate(prompt, max_new_tokens=50, seed=42, temperature=0.8, top_k=50)

print("Run 1:", text1[:100], "...")
print("Run 2:", text2[:100], "...")
print(f"\nOutputs match: {text1 == text2}")

## Batch generation

In [ ]:
prompts = [
    "The theory of relativity states that",
    "In computer science, a hash table is",
    "The best way to learn programming is",
]

results = engine.generate_batch(prompts, max_new_tokens=60, temperature=0.8, top_k=50)

for i, text in enumerate(results):
    print(f"--- Prompt {i+1} ---")
    print(text)
    print()